# Учебное исследование: Октябрьский район Барнаула

Этот ноутбук по шагам выполняет городское исследование с помощью библиотеки **BlocksNet** (версия 1.0.0а9):

1. скачивает открытые данные **OpenStreetMap** по границам Октябрьского района Барнаула;
2. нарезает территорию на **кварталы** и определяет их **землепользование**;
3. строит **граф передвижения пешком** и считает **время в пути** между кварталами (в минутах);
4. оценивает **население** по зданиям и агрегирует их к кварталам;
5. считает городские **показатели**: обеспеченность школами, плотность застройки, морфотипы застройки, доступность территории.

---

**Как запускать.** Ячейки выполняются по порядку сверху вниз: выберите ячейку и нажмите `Shift + Enter`.

- Первый запуск самый долгий: скачивание данных из OpenStreetMap и расчёт матрицы времени занимают ориентировочно **10–30 минут**.
- Повторные запуски быстрые: готовые данные автоматически берутся из папки `data/barnaul/`.
- Если ячейка завершилась ошибкой из-за интернета — просто запустите её ещё раз.

**Как открыть ноутбук.**

- Вариант A (рекомендуется) — VS Code: откройте папку проекта, затем файл `examples/barnaul_blocksnet.ipynb`; в правом верхнем углу нажмите `Select Kernel` и выберите окружение **.venv**.
- Вариант B — браузер: откройте терминал в папке проекта и выполните команды:

```bash
cd "D:/Рабочий стол/3 семестр/ТИИ/python-package-boilerplate"
.venv/Scripts/python.exe -m jupyter lab
```

затем откройте в браузере `examples/barnaul_blocksnet.ipynb`.

**Главные настройки** (территория, способ передвижения, папка данных) задаются в первой ячейке ниже.

## 0. Настройки и служебные функции

Сначала задаются главные настройки исследования, а затем определяются небольшие вспомогательные функции.
Они сохраняют все слои в папку `data/barnaul/` и умеют загружать их обратно — благодаря этому повторный запуск ноутбука не скачивает данные заново.

In [2]:
# ─── Библиотеки ───────────────────────────────────────────────────
import pickle
from pathlib import Path

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import matplotlib.pyplot as plt

import blocksnet

# ─── Главные настройки исследования ───────────────────────────────
TERRITORY = 'Октябрьский район, Барнаул, Алтайский край'   # территория исследования
GRAPH_TYPE = 'walk'            # способ передвижения: 'walk' — пешком, 'drive' — на автомобиле
DATA_DIR = Path('data') / 'barnaul'    # папка, куда складываются все данные и результаты

DATA_DIR.mkdir(parents=True, exist_ok=True)

print('blocksnet:', blocksnet.__version__, '| osmnx:', ox.__version__)
print('geopandas:', gpd.__version__, '| pandas:', pd.__version__)
print('Папка данных:', DATA_DIR.resolve())

blocksnet: 1.0.0a9 | osmnx: 2.0.7
geopandas: 1.1.4 | pandas: 2.3.3
Папка данных: D:\Рабочий стол\3 семестр\ТИИ\python-package-boilerplate\examples\data\barnaul


In [3]:
# ─── Служебные функции: сохранение и загрузка данных ──────────────

def _to_text(v):
    """Превращает значение в текст (в том числе перечисления, например LandUse); пустые остаются пустыми."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return str(getattr(v, 'value', v))


def save_geojson(gdf, name):
    """Сохраняет географический слой в файл data/barnaul/<name>.geojson (в градусах, EPSG:4326)."""
    path = DATA_DIR / f'{name}.geojson'
    out = gdf.copy()
    for col in out.columns:
        if col != out.geometry.name and out[col].dtype == object:
            out[col] = out[col].map(_to_text)
    out.to_crs(4326).to_file(path, driver='GeoJSON')
    print(f'Сохранено: {path.name} ({len(out)} объектов)')
    return path


def load_geojson(name, crs=None):
    """Загружает слой из data/barnaul/<name>.geojson, если файл уже есть. Иначе возвращает None."""
    path = DATA_DIR / f'{name}.geojson'
    if not path.exists():
        return None
    gdf = gpd.read_file(path)
    if crs is not None:
        gdf = gdf.to_crs(crs)      # обратно в метрическую систему (метры)
    print(f'Загружено из файла: {path.name} ({len(gdf)} объектов)')
    return gdf


def save_csv(df, name):
    """Сохраняет таблицу в файл data/barnaul/<name>.csv."""
    path = DATA_DIR / f'{name}.csv'
    df.to_csv(path)
    print(f'Сохранено: {path.name}')
    return path


def load_csv(name):
    """Загружает таблицу из data/barnaul/<name>.csv, если файл уже есть. Иначе возвращает None."""
    path = DATA_DIR / f'{name}.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path, index_col=0)
    print(f'Загружено из файла: {path.name}')
    return df


def save_graph(graph, name):
    """Сохраняет граф дорожной сети в файл data/barnaul/<name>.pkl."""
    path = DATA_DIR / f'{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(graph, f)
    print(f'Сохранено: {path.name}')
    return path


def load_graph(name):
    """Загружает граф из data/barnaul/<name>.pkl, если файл уже есть. Иначе возвращает None."""
    path = DATA_DIR / f'{name}.pkl'
    if not path.exists():
        return None
    with open(path, 'rb') as f:
        graph = pickle.load(f)
    print(f'Граф загружен из файла: {path.name}')
    return graph


print('Служебные функции готовы.')

Служебные функции готовы.


## 1. Границы территории и данные OpenStreetMap

**Границы** района берутся из OpenStreetMap автоматически — по названию территории (геокодер). Если район по названию не найдётся, контур можно задать вручную: подсказка находится в комментариях к ячейке.

Затем скачиваются три слоя инфраструктуры: **улично-дорожная сеть**, **железные дороги** и **вода** (реки, озёра, каналы). Источник данных — сервис Overpass (OpenStreetMap).

Скачивание идёт в географических координатах (градусы), а все измерения библиотека выполняет в метрической системе (метры), которая подбирается автоматически под нашу территорию.

In [ ]:
# ─── Шаг 1.1. Границы территории ──────────────────────────────────
boundaries = load_geojson('boundaries')

if boundaries is None:
    # Границы ищем в OpenStreetMap по названию.
    # Если район не найдётся, можно задать его напрямую, например:
    #   ox.geocode_to_gdf('R1234567')  — по номеру объекта OSM (отношения),
    #   либо взять контур города целиком: 'Барнаул, Алтайский край'.
    boundaries = ox.geocode_to_gdf(TERRITORY)[['geometry']].copy()
    save_geojson(boundaries, 'boundaries')

crs = boundaries.estimate_utm_crs()     # подходящая метрическая система координат
boundaries = boundaries.to_crs(crs)     # переводим границы в метры

print('Система координат для измерений:', crs.name)
boundaries.plot(color='#d9ead3', edgecolor='#38761d', figsize=(8, 8)).set_axis_off()
plt.title('Границы территории исследования');

Ниже задаются наборы тегов OpenStreetMap и скачиваются слои. Если файлы уже есть в `data/barnaul/`, скачивание пропускается и данные берутся с диска.

In [ ]:
# ─── Шаг 1.2. Дороги, железные дороги и вода ──────────────────────
bc_tags = {
    'roads': {
        'highway': ['construction', 'crossing', 'living_street', 'motorway', 'motorway_link',
                    'motorway_junction', 'pedestrian', 'primary', 'primary_link', 'raceway',
                    'residential', 'road', 'secondary', 'secondary_link', 'services', 'tertiary',
                    'tertiary_link', 'track', 'trunk', 'trunk_link', 'turning_circle',
                    'turning_loop', 'unclassified'],
        'service': ['living_street', 'emergency_access'],
    },
    'railways': {
        'railway': 'rail',
    },
    'water': {
        'riverbank': True,
        'reservoir': True,
        'basin': True,
        'dock': True,
        'canal': True,
        'pond': True,
        'natural': ['water', 'bay'],
        'waterway': ['river', 'canal', 'ditch'],
        'landuse': 'basin',
        'water': 'lake',
    },
}

water = load_geojson('water', crs=crs)
roads = load_geojson('roads', crs=crs)
railways = load_geojson('railways', crs=crs)

if water is None or roads is None or railways is None:
    print('Скачиваем данные из OpenStreetMap (Overpass), это может занять несколько минут...')
    area = boundaries.to_crs(4326).union_all()
    water = ox.features_from_polygon(area, bc_tags['water'])
    roads = ox.features_from_polygon(area, bc_tags['roads'])
    railways = ox.features_from_polygon(area, bc_tags['railways'])

    # оставляем только нужные типы геометрий
    water = water[water.geom_type.isin(['Polygon', 'MultiPolygon', 'LineString', 'MultiLineString'])]
    roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])]
    railways = railways[railways.geom_type.isin(['LineString', 'MultiLineString'])]

    save_geojson(water, 'water')
    save_geojson(roads, 'roads')
    save_geojson(railways, 'railways')

print('Вода:', len(water), '| Дороги:', len(roads), '| Железные дороги:', len(railways))

In [ ]:
# ─── Шаг 1.3. Визуальная проверка исходных слоёв ──────────────────
fig, ax = plt.subplots(figsize=(8, 8))
roads.plot(ax=ax, color='#999999', linewidth=0.5, label='дороги')
railways.plot(ax=ax, color='#333333', linewidth=1, label='железные дороги')
water.plot(ax=ax, color='#9fc5e8', label='вода')
boundaries.boundary.plot(ax=ax, color='#38761d', linewidth=2, label='граница')
ax.set_axis_off()
ax.set_title('Исходные данные OpenStreetMap')
ax.legend(loc='lower right', fontsize=8);

## 2. Кварталы: нарезка и землепользование

**Кварталы** — небольшие участки территории, ограниченные улицами, железными дорогами и водой. BlocksNet нарезает их автоматически из скачанных слоёв: каждый квартал будет отдельной строкой в таблице результатов.

**Землепользование** — тип использования квартала (жилой, деловой, промышленный, рекреация и т. д.). Оно определяется по зонам OpenStreetMap (`landuse`) с помощью словаря правил `rules`: при необходимости словарь можно дополнить своими зонами.

In [ ]:
# ─── Шаг 2.1. Нарезка кварталов ───────────────────────────────────
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

blocks = load_geojson('blocks', crs=crs)

if blocks is None:
    lines, polygons = preprocess_urban_objects(roads, railways, water)
    blocks = cut_urban_blocks(boundaries, lines, polygons)

print('Кварталов:', len(blocks))

blocks.plot(color='#fce5cd', edgecolor='#e69138', linewidth=0.4, figsize=(8, 8)).set_axis_off()
plt.title(f'Кварталы территории ({len(blocks)} шт.)');

In [ ]:
# ─── Шаг 2.2. Землепользование кварталов ──────────────────────────
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.enums import LandUse

# Правила: какая зона OpenStreetMap какому типу землепользования соответствует
rules = {
    'commercial': LandUse.BUSINESS,
    'industrial': LandUse.INDUSTRIAL,
    'cemetery': LandUse.SPECIAL,
    'garages': LandUse.INDUSTRIAL,
    'residential': LandUse.RESIDENTIAL,
    'retail': LandUse.BUSINESS,
    'grass': LandUse.RECREATION,
    'farmland': LandUse.AGRICULTURE,
    'construction': LandUse.SPECIAL,
    'brownfield': LandUse.INDUSTRIAL,
    'forest': LandUse.RECREATION,
    'recreation_ground': LandUse.RECREATION,
    'religious': LandUse.SPECIAL,
    'flowerbed': LandUse.RECREATION,
    'military': LandUse.SPECIAL,
    'landfill': LandUse.TRANSPORT,
}

if 'land_use' not in blocks.columns:

    # зоны OpenStreetMap (скачиваем один раз, затем берём из файла)
    functional_zones = load_geojson('functional_zones', crs=crs)
    if functional_zones is None:
        functional_zones = ox.features_from_polygon(
            boundaries.to_crs(4326).union_all(),
            tags={'landuse': True},
        )
        functional_zones = functional_zones.reset_index(drop=True)[['geometry', 'landuse']]
        functional_zones = functional_zones.rename(columns={'landuse': 'functional_zone'})
        functional_zones = functional_zones.to_crs(crs)
        save_geojson(functional_zones, 'functional_zones')

    blocks = assign_land_use(blocks, functional_zones, rules)
    save_geojson(blocks, 'blocks')     # теперь кварталы сохранены вместе с землепользованием

print(blocks['land_use'].value_counts(dropna=False).to_string())
print()
print('Кварталов без данных о землепользовании:', int(blocks['land_use'].isna().sum()))

ax = blocks.plot(color='#dddddd', figsize=(8, 8))
blocks.plot(column='land_use', ax=ax, legend=True)
ax.set_axis_off()
ax.set_title('Землепользование кварталов');

---

# ⛔ ОСТАНОВКА: подготовка кварталов завершена

**Самое время остановиться и проверить результат перед расчётами.**

Что уже сделано:
1. получены границы территории и слои дорог, железных дорог и воды из OpenStreetMap;
2. территория нарезана на кварталы;
3. определено землепользование кварталов.

**Что стоит проверить:**
- карта кварталов покрывает весь район (очень крупные кварталы в лесу, парке или у воды — это нормально);
- число кварталов для городского района обычно составляет несколько сотен;
- у большинства кварталов есть тип землепользования; там, где в OpenStreetMap нет данных, значение будет пустым — это нормально.

**Дальше библиотека BlocksNet может посчитать по этой территории:**
1. **время в пути между кварталами пешком** — матрица доступности в минутах (граф из OpenStreetMap; для автомобиля меняется одна настройка `GRAPH_TYPE`);
2. **население и жилой фонд** — оценка по зданиям OpenStreetMap, агрегированная к кварталам;
3. **обеспеченность школами** — с учётом распределения мест между кварталами и норматива доступности;
4. **плотностные показатели застройки** — FSI, GSI, MXI, этажность, доля озеленения и жилых площадей;
5. **морфотипы застройки** — автоматическая классификация кварталов (методы Spacematrix и Strelka);
6. **доступность территории** — средневзвешенное время в пути между кварталами района.

**Перед запуском следующей части согласуйте список показателей.** Если какой-то показатель не нужен — его ячейки можно просто не запускать. Если нужны другие услуги (детские сады, магазины, поликлиники и т. п.) — это выясняется здесь, до расчётов, потому что от этого зависит, какие слои скачивать.

После согласования просто продолжайте выполнять ячейки дальше по порядку.

---

## 3. Граф передвижения и время в пути (пешком)

**Граф передвижения** — это сеть улиц района, по которой строятся маршруты. Библиотека скачивает её из OpenStreetMap (готовые графы: `walk` — пешеходный, `drive` — автомобильный). Мы используем значение настройки `GRAPH_TYPE` — по умолчанию **пешком**.

**Время в пути** считается между центрами кварталов по кратчайшим путям и выражается в **минутах** (вес каждого участка улицы — время его прохода). Результат — квадратная таблица: «**матрица доступности**», где строки и столбцы — номера кварталов, а значения — время в пути в минутах.

При первом скачивании может появиться предупреждение о системе координат (CRS) — это нормально: библиотека сама перепроецирует данные, как ей нужно.

In [ ]:
# ─── Шаг 3.1. Граф передвижения ───────────────────────────────────
from blocksnet.relations import get_accessibility_graph

graph_name = f'{GRAPH_TYPE}_graph'      # например, walk_graph
graph = load_graph(graph_name)

if graph is None:
    print('Скачиваем граф передвижения из OpenStreetMap...')
    graph = get_accessibility_graph(boundaries, GRAPH_TYPE)
    save_graph(graph, graph_name)

print('Узлов (перекрёстков):', graph.number_of_nodes())
print('Рёбер (участков улиц):', graph.number_of_edges())

In [ ]:
# ─── Шаг 3.2. Матрица времени между кварталами ────────────────────
from blocksnet.relations import calculate_accessibility_matrix

mx_name = f'acc_mx_{GRAPH_TYPE}'        # например, acc_mx_walk
acc_mx = load_csv(mx_name)

if acc_mx is None:
    print('Считаем время в пути между всеми парами кварталов (это может занять несколько минут)...')
    acc_mx = calculate_accessibility_matrix(blocks, graph)
    save_csv(acc_mx, mx_name)
else:
    # номера кварталов в шапке файла хранятся текстом — превращаем их обратно в числа
    acc_mx.columns = acc_mx.columns.astype(int)

print('Размер матрицы (число кварталов):', acc_mx.shape[0], 'x', acc_mx.shape[1])
print('Все значения — время в пути в минутах.')

In [ ]:
# ─── Шаг 3.3. Проверка матрицы времени ────────────────────────────
values = acc_mx.values
finite_values = values[np.isfinite(values)]

print('Диагональ нулевая (квартал к себе):', np.allclose(np.diag(values), 0))
print('Минимум (не считая нуля), мин:', round(float(np.min(finite_values[finite_values > 0])), 1))
print('Медиана, мин:', round(float(np.median(finite_values)), 1))
print('Максимум среди связанных пар, мин:', round(float(np.max(finite_values)), 1))
print('Несвязанных пар (острова/изолированные участки):', int(np.isinf(values).sum()))

pd.Series(finite_values).hist(bins=40, color='#6aa84f')
plt.xlabel('Время в пути, минуты')
plt.ylabel('Число пар кварталов')
plt.title('Распределение времени в пути между кварталами пешком');


## 4. Здания, жилой фонд и население

Скачиваем здания из OpenStreetMap и оцениваем по ним **население**.

Как это устроено в BlocksNet:
- здание считается **жилым**, если его тег `building` указывает на жильё (residential, house, apartments, ...);
- **этажность** берётся из тега `building:levels`, если он указан;
- остальные характеристики досчитываются моделью: площадь застройки — по геометрии, общая площадь — площадь × этажность, жилая площадь — 90 % общей, население — примерно один человек на 30 м² жилой площади (`default_living_demand=30`).

Затем здания **агрегируются к кварталам**: для каждого квартала суммируются площади и население. Именно эти числа пойдут в показатели.

In [ ]:
# ─── Шаг 4.1. Здания из OpenStreetMap ─────────────────────────────
from blocksnet.preprocessing.imputing import impute_buildings

buildings = load_geojson('buildings', crs=crs)

if buildings is None:
    print('Скачиваем здания из OpenStreetMap (может занять несколько минут)...')
    buildings = ox.features_from_polygon(
        boundaries.to_crs(4326).union_all(),
        tags={'building': True},
    )
    buildings = buildings.reset_index(drop=True).to_crs(crs)

    # 1) какие здания жилые — по тегу building
    is_living_tags = ['residential', 'house', 'apartments', 'detached', 'terrace', 'dormitory']
    if 'building' in buildings.columns:
        buildings['is_living'] = buildings['building'].isin(is_living_tags)
    else:
        buildings['is_living'] = False

    # 2) этажность — по тегу building:levels (если тег есть)
    if 'building:levels' in buildings.columns:
        buildings['number_of_floors'] = pd.to_numeric(buildings['building:levels'], errors='coerce')
    else:
        buildings['number_of_floors'] = np.nan

    # 3) досчитываем площади и население моделью BlocksNet
    buildings = impute_buildings(buildings, default_living_demand=30)
    save_geojson(buildings, 'buildings')

print('Зданий:', len(buildings))
print('Оценка населения района, чел.:', int(buildings['population'].sum()))

In [ ]:
# ─── Шаг 4.2. Агрегация зданий к кварталам ────────────────────────
from blocksnet.blocks.aggregation import aggregate_objects

buildings_blocks = aggregate_objects(blocks, buildings)[0]
save_geojson(buildings_blocks, 'buildings_blocks')

print('Население в границах кварталов, чел.:', int(buildings_blocks['population'].sum()))

buildings_blocks.plot('population', legend=True, figsize=(8, 8)).set_axis_off()
plt.title('Население по кварталам, чел.');

## 5. Услуги: школы

Аналогично зданиям, скачиваем социальный объект — **школы** (`amenity=school`). У каждой школы важно число мест (`capacity`): если в OpenStreetMap оно не указано, BlocksNet подставляет типовое значение из своей конфигурации.

Затем школы агрегируются к кварталам. Показатель обеспеченности из следующего раздела учитывает, как места распределяются между кварталами-соседями с учётом конкуренции — это честнее, чем просто считать школы в радиусе.

In [ ]:
# ─── Шаг 5.1. Школы из OpenStreetMap ──────────────────────────────
from blocksnet.preprocessing.imputing import impute_services

schools = load_geojson('schools', crs=crs)

if schools is None:
    print('Скачиваем школы из OpenStreetMap...')
    schools = ox.features_from_polygon(
        boundaries.to_crs(4326).union_all(),
        tags={'amenity': 'school'},
    )
    schools = schools.reset_index(drop=True).to_crs(crs)

    # вместимость из тега capacity (строки вида '450' превращаем в числа, непонятные значения — в пустые)
    if 'capacity' in schools.columns:
        schools['capacity'] = pd.to_numeric(schools['capacity'], errors='coerce')

    schools = impute_services(schools, 'school')    # досчитываем недостающую вместимость
    save_geojson(schools, 'schools')

schools_blocks = aggregate_objects(blocks, schools)[0]
save_geojson(schools_blocks, 'schools_blocks')

print('Школ:', len(schools), '| Суммарная вместимость, мест:', int(schools['capacity'].sum()))

schools_blocks.plot('capacity', legend=True, figsize=(8, 8)).set_axis_off()
plt.title('Вместимость школ по кварталам, мест');

## 6. Городские показатели

Здесь считаются показатели исследования. Сначала — **обеспеченность школами**.

Как это устроено: для каждого квартала по числу жителей считается **спрос** (норматив BlocksNet для школ — 120 мест на 1000 жителей). Затем алгоритм распределяет места школ между кварталами-соседями в пределах нормативной доступности (15 минут) с учётом конкуренции. Результат:
- **provision_strong** — доля спроса, покрытая местами (1.0 = норматив выполнен; для кварталов без жителей значение не определяется);
- **provision_weak** — более мягкая оценка (учитывает и более дальние школы).

In [ ]:
# ─── Шаг 6.1. Обеспеченность школами ──────────────────────────────
from blocksnet.config import service_types_config
from blocksnet.analysis.provision import competitive_provision

cfg = service_types_config['school']
print('Норматив по школам:', cfg['demand'], 'мест на 1000 жителей;',
      'нормативная доступность:', cfg['accessibility'], 'минут')

df = buildings_blocks[['population']].join(schools_blocks[['capacity']])
prov_df, links_df = competitive_provision(df, acc_mx, cfg['accessibility'], cfg['demand'])

save_csv(prov_df, 'provision_school')
save_csv(links_df, 'provision_school_links')

ax = blocks.plot(color='#dddddd', figsize=(8, 8))
blocks.join(prov_df).plot('provision_strong', ax=ax, cmap='RdYlGn', vmin=0, vmax=1, legend=True)
ax.set_axis_off()
ax.set_title('Обеспеченность школами (1 — норматив выполнен, красный — дефицит)')

need = prov_df[prov_df['demand'] > 0]
print('Кварталов с потребностью в школах:', len(need))
print('Из них обеспечены менее чем наполовину:', int((need['provision_strong'] < 0.5).sum()))

### Плотность застройки

Для каждого квартала считаются плотностные показатели:

- **FSI** (коэффициент плотности застройки) — общая площадь всех зданий / площадь квартала; чем ближе к центру города, тем обычно больше;
- **GSI** (коэффициент застроенности) — площадь под зданиями / площадь квартала;
- **MXI** (интенсивность использования) — общая площадь зданий на одного жителя, м²/чел.;
- **L** — средневзвешенная этажность;
- **OSR** — доля открытых, свободных от застройки пространств;
- **share_living / share_non_living** — доли жилых и нежилых площадей.

In [ ]:
# ─── Шаг 6.2. Плотность застройки ─────────────────────────────────
from blocksnet.analysis.indicators import calculate_density_indicators

buildings_blocks['site_area'] = buildings_blocks.area      # площадь квартала, м²
density_df = calculate_density_indicators(buildings_blocks)
save_csv(density_df, 'density')

density_df[['fsi', 'gsi', 'mxi', 'l', 'osr', 'share_living']].describe().round(2)

In [ ]:
# карта плотности застройки
ax = blocks.plot(color='#dddddd', figsize=(8, 8))
blocks.join(density_df).plot('fsi', ax=ax, legend=True, vmin=0, vmax=3)
ax.set_axis_off()
ax.set_title('Плотность застройки FSI (общая площадь зданий / площадь квартала)');

### Морфотипы застройки

BlocksNet умеет автоматически **классифицировать застройку кварталов** — двумя независимыми методами:

- **Spacematrix** — делит кварталы на классы по сочетанию плотности (FSI) и интенсивности использования (MXI);
- **Strelka** — классификация российского исследовательского института «Стрелка» (на основе этажности, плотности и интенсивности).

Результат — метки (morphotype) для каждого квартала; сравнение двух методов покажет, где они согласны.

In [ ]:
# ─── Шаг 6.3. Морфотипы Spacematrix ───────────────────────────────
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes

spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df)
save_csv(spacematrix_df, 'spacematrix')

ax = blocks.plot(color='#dddddd', figsize=(8, 8))
blocks.join(spacematrix_df).plot('morphotype', ax=ax, legend=True)
ax.set_axis_off()
ax.set_title('Морфотипы застройки, метод Spacematrix')

spacematrix_df['morphotype'].value_counts().to_string()

In [ ]:
# ─── Шаг 6.4. Морфотипы Strelka ───────────────────────────────────
from blocksnet.analysis.morphotypes import get_strelka_morphotypes

strelka_df = get_strelka_morphotypes(density_df)
save_csv(strelka_df, 'strelka')

ax = blocks.plot(color='#dddddd', figsize=(8, 8))
blocks.join(strelka_df).plot('morphotype', ax=ax, legend=True)
ax.set_axis_off()
ax.set_title('Морфотипы застройки, метод Strelka')

strelka_df['morphotype'].value_counts().to_string()

### Доступность территории

Последний показатель — **area_accessibility**: средневзвешенное время в пути от квартала до всех остальных кварталов района (в минутах). Чем меньше значение, тем удобнее расположение квартала. Весом служит площадь кварталов: считается, что до крупных кварталов «доезжает» больше значимости.

In [ ]:
# ─── Шаг 6.5. Доступность территории ──────────────────────────────
from blocksnet.analysis.network import area_accessibility

# Заменяем бесконечности (несвязанные островки) на максимальное реальное время пути
acc_mx_clean = acc_mx.copy()
finite_vals = acc_mx_clean.values[np.isfinite(acc_mx_clean.values)]
max_finite = float(np.max(finite_vals)) if len(finite_vals) > 0 else 60.0
acc_mx_clean = acc_mx_clean.replace(np.inf, max_finite)

area_acc = area_accessibility(acc_mx_clean, buildings_blocks)
save_csv(area_acc, 'area_accessibility')

blocks.join(area_acc).plot('area_accessibility', legend=True, figsize=(8, 8)).set_axis_off()
plt.title('Доступность территории (среднее время в пути, мин)');


## 7. Итоговая таблица

Собираем все показатели в один слой: каждый квартал — строка, показатели — столбцы. Результат сохраняется в двух форматах:

- `blocks_all_indicators.geojson` — геометрия кварталов вместе с показателями (для карт);
- `blocks_all_indicators.csv` — та же таблица без геометрии (для анализа в Excel).

In [ ]:
# ─── Шаг 7.1. Сводная таблица результатов ─────────────────────────
blocks_all = (
    blocks
    .join(prov_df[['provision_strong', 'provision_weak']])
    .join(density_df[['fsi', 'gsi', 'mxi', 'l', 'osr', 'share_living', 'share_non_living']])
    .join(spacematrix_df[['cluster', 'morphotype']].rename(
        columns={'cluster': 'spacematrix_cluster', 'morphotype': 'spacematrix_morphotype'}))
    .join(strelka_df[['morphotype']].rename(columns={'morphotype': 'strelka_morphotype'}))
    .join(area_acc)
)

save_geojson(blocks_all, 'blocks_all_indicators')
blocks_all.drop(columns='geometry').to_csv(DATA_DIR / 'blocks_all_indicators.csv')
print('Таблица сохранена: строк (кварталов):', len(blocks_all), ', столбцов:', blocks_all.shape[1])

In [ ]:
# ─── Шаг 7.2. Краткие итоги ───────────────────────────────────────
print('ИТОГИ ЭТАПА 1: город Барнаул')
print('Кварталов:', len(blocks_all))
print('Население (оценка по зданиям OSM), чел.:', int(buildings_blocks['population'].sum()))
print('Школ:', len(schools), '| вместимость, мест:', int(schools['capacity'].sum()))
print('Медианное время в пути пешком между кварталами, мин:',
      round(float(np.nanmedian(acc_mx.values)), 1))
print('Кварталов с потребностью в школах:', len(need))
print('  обеспечены на 50% и лучше:', int((need['provision_strong'] >= 0.5).sum()))
print('  обеспечены полностью (100%):', int((need['provision_strong'] >= 1.0).sum()))
print('Средняя плотность застройки FSI:', round(float(density_df['fsi'].mean()), 2))
print()
print('Файлы в папке данных:')
for p in sorted(DATA_DIR.iterdir()):
    print(f'  {p.name:36s} {p.stat().st_size / 1024 / 1024:8.2f} МБ')

## Итоги и что дальше

### Чек-лист проверки результатов

- [ ] карта кварталов покрывает весь район; крупные кварталы в лесу или у воды — это нормально;
- [ ] матрица времени квадратная, диагональ нулевая, значения — минуты (медиана в разумных пределах, например 10–40 минут);
- [ ] население района выглядит правдоподобно (обычно десятки–сотни тысяч человек для городского района);
- [ ] у большинства кварталов с потребностью обеспеченность школами выше 0; красные кварталы на карте — это дефицит мест;
- [ ] морфотипы разных методов осмысленно накладываются на карту застройки;
- [ ] в папке `data/barnaul/` появился файл `blocks_all_indicators.geojson`.

### Повторный запуск

- Просто выполняйте ячейки сверху вниз: всё, что уже посчитано, берётся из файлов `data/barnaul/` и пересчитывается за секунды.
- **Полная пересборка**: удалите папку `examples/data/barnaul` и запустите ноутбук заново (снова понадобится интернет).
- **Другая территория**: поменяйте `TERRITORY` в первой ячейке, удалите папку данных и запустите ноутбук заново.
- **Другая скорость передвижения**: поменяйте `GRAPH_TYPE` на `'drive'` — граф и матрица скачаются и посчитаются под новым именем, старые файлы не пострадают.
- Если скачивание из OpenStreetMap прервалось из-за сети — просто запустите ячейку ещё раз.

### Что дальше (этап 2)

По сохранённым данным (`blocks_all_indicators.geojson` и файлам CSV) будет построена одна HTML-страница с интерактивной картой и дашбордом: те же показатели, графики, таблицы и краткие выводы.